# Dependency Distance ZA Distribution Fitting

## Experiment Overview

This notebook implements the full experimental pipeline for analyzing dependency distance distributions across Universal Dependencies (UD) treebanks:

1. **Load and merge** dependency distance data with WALS typological features
2. **Fit Right Truncated Modified Zipf-Alekseev (ZA) distributions** to dependency distance data from each treebank
3. **Analyze spoken vs. written** differences in dependency distance
4. **Run mixed-effects models** to predict ZA parameters from WALS features
5. **Detect outliers** via random effects
6. **Perform robustness checks** (bootstrap, distribution comparison)

## Background

Dependency distance minimization is a key principle in syntax. This experiment investigates:
- Whether spoken language minimizes dependency distance more than written language
- How typological features (from WALS) interact with dependency distance patterns
- Which language families deviate from global patterns

## Data Sources

- **Universal Dependencies (UD)**: Dependency-annotated treebanks from 350+ languages
- **WALS (World Atlas of Language Structures)**: Typological features for linguistic characterization

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'statsmodels==0.14.6')

print('Dependencies installed successfully')

In [ ]:
# Imports - copied from original method.py
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from scipy import optimize, stats
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
import warnings
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings('ignore')

# For visualization at the end
import matplotlib.pyplot as plt

print('Imports successful')

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6488ab-typological-predictors-of-dependency-dis/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined')

In [ ]:
# Load the demo data
data = load_data()
print(f"Data loaded successfully")
print(f"DD examples: {len(data.get('dd_examples', []))}")
print(f"WALS examples: {len(data.get('wals_examples', []))}")

In [ ]:
# Configuration - set to ABSOLUTE MINIMUM values for demo
# These can be scaled up once the demo runs successfully

# Minimum dependencies per treebank for inclusion
MIN_DEPS = 10  # Original: 1000

# Number of bootstrap iterations for robustness checks
N_BOOTSTRAP = 10  # Original: 100

# Maximum iterations for ZA distribution fitting
MAXITER_FIT = 100  # Original: 1000

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Configuration set (MINIMUM values for demo)')

## Step 1: Helper Functions

These functions parse the input JSON and extract examples from the dataset format.

In [ ]:
# Helper functions - from method.py

def parse_input_json(input_str: str) -> Dict:
    """Parse input JSON string from example."""
    try:
        return json.loads(input_str)
    except:
        return {}

def parse_output_value(output_str: str) -> int:
    """Parse output value (dependency distance)."""
    try:
        return int(output_str)
    except:
        return 0

print('Helper functions defined')

## Step 2: Merge Datasets

Merge dependency distance data with WALS typological features on treebank name.
This creates a unified DataFrame with DD observations and their corresponding WALS features.

In [ ]:
# Merge datasets function - from method.py

def merge_datasets(dd_examples: List[Dict], wals_examples: List[Dict]) -> pd.DataFrame:
    """Merge dependency distance data with WALS mappings."""
    print(f"Merging datasets...")
    
    # Process DD examples
    dd_rows = []
    for ex in dd_examples:
        input_data = parse_input_json(ex.get('input', '{}'))
        dd = parse_output_value(ex.get('output', '0'))
        
        row = {
            'treebank_name': input_data.get('treebank_name', ''),
            'language': input_data.get('language', ''),
            'family': input_data.get('family', ''),
            'genre': input_data.get('genre', ''),
            'sentence_length': input_data.get('sentence_length', 0),
            'deprel': input_data.get('deprel', ''),
            'head_position': input_data.get('head_position', 0),
            'dependent_position': input_data.get('dependent_position', 0),
            'dd': dd
        }
        dd_rows.append(row)
    
    dd_df = pd.DataFrame(dd_rows)
    print(f"DD DataFrame shape: {dd_df.shape}")
    print(f"Unique treebanks in DD data: {dd_df['treebank_name'].nunique()}")
    
    # Process WALS examples
    wals_rows = []
    for ex in wals_examples:
        # Parse the output which contains wals_features
        output_str = ex.get('output', '{}')
        if isinstance(output_str, str):
            try:
                output_data = json.loads(output_str)
            except:
                output_data = {}
        else:
            output_data = output_str
        
        wals_features = output_data.get('wals_features', {})
        
        row = {
            'treebank_name': ex.get('metadata_treebank', ''),
            'wals_1A_value': wals_features.get('1A', {}).get('value', 'NA'),
            'wals_20A_value': wals_features.get('20A', {}).get('value', 'NA'),
            'wals_26A_value': wals_features.get('26A', {}).get('value', 'NA'),
            'wals_49A_value': wals_features.get('49A', {}).get('value', 'NA'),
            'wals_51A_value': wals_features.get('51A', {}).get('value', 'NA'),
        }
        wals_rows.append(row)
    
    wals_df = pd.DataFrame(wals_rows)
    print(f"WALS DataFrame shape: {wals_df.shape}")
    print(f"Unique treebanks in WALS data: {wals_df['treebank_name'].nunique()}")
    
    # Merge on treebank_name
    merged_df = dd_df.merge(wals_df, on='treebank_name', how='inner')
    print(f"Merged DataFrame shape: {merged_df.shape}")
    print(f"Unique treebanks after merge: {merged_df['treebank_name'].nunique()}")
    
    return merged_df

print('merge_datasets function defined')

## Step 3: Filter Treebanks

Filter to treebanks with minimum number of dependencies and complete WALS data.

In [ ]:
# Filter treebanks function - from method.py

def filter_treebanks(merged_df: pd.DataFrame, min_deps: int = 1000) -> pd.DataFrame:
    """Filter to treebanks with minimum number of dependencies and complete WALS data."""
    print(f"Filtering treebanks with at least {min_deps} dependencies...")
    
    # Count dependencies per treebank
    treebank_counts = merged_df.groupby('treebank_name').size().reset_index(name='n_deps')
    
    # Filter by min_deps
    valid_treebanks = treebank_counts[treebank_counts['n_deps'] >= min_deps]['treebank_name'].tolist()
    print(f"Treebanks with >= {min_deps} deps: {len(valid_treebanks)}")
    
    filtered_df = merged_df[merged_df['treebank_name'].isin(valid_treebanks)].copy()
    
    # Filter to complete WALS data (no NA values)
    wals_cols = ['wals_1A_value', 'wals_20A_value', 'wals_26A_value', 'wals_49A_value', 'wals_51A_value']
    complete_mask = True
    for col in wals_cols:
        complete_mask = complete_mask & (filtered_df[col] != 'NA') & (filtered_df[col].notna())
    
    complete_df = filtered_df[complete_mask].copy()
    print(f"Treebanks with complete WALS data: {complete_df['treebank_name'].nunique()}")
    
    return complete_df

print('filter_treebanks function defined')

## Step 4: ZA Distribution Fitting

Fit Right Truncated Modified Zipf-Alekseev distributions to dependency distance data from each treebank.

In [ ]:
# ZA distribution functions - from method.py

def compute_dd_distribution(treebank_data: pd.DataFrame) -> Dict[int, int]:
    """Compute empirical distribution of dependency distances for a treebank."""
    return treebank_data['dd'].value_counts().to_dict()

def neg_log_lik_zipf_alekseev(params: Tuple[float, float], dd_counts: Dict[int, int], L: int) -> float:
    """Negative log-likelihood for Right Truncated Modified Zipf-Alekseev distribution."""
    a, b = params
    if not (np.isfinite(a) and np.isfinite(b)):
        return 1e10
    x_vals = np.arange(1, L + 1)
    try:
        log_unnorm = -(a + b * np.log(x_vals)) * np.log(x_vals)
        unnorm_p = np.exp(log_unnorm)
    except:
        return 1e10
    if not np.all(np.isfinite(unnorm_p)) or np.sum(unnorm_p) <= 0:
        return 1e10
    p = unnorm_p / np.sum(unnorm_p)
    ll = 0.0
    for x, count in dd_counts.items():
        if 1 <= x <= L and p[x-1] > 0:
            ll += count * np.log(p[x-1])
    return -ll

def fit_za_distribution(dd_counts: Dict[int, int]) -> Dict:
    """Fit Right Truncated Modified Zipf-Alekseev distribution using MLE."""
    if not dd_counts or len(dd_counts) < 2:
        return {'a_param': np.nan, 'b_param': np.nan, 'converged': False}
    L = max(dd_counts.keys())
    total_count = sum(dd_counts.values())
    initial_params = [1.0, 0.0]
    bounds = [(-10, 10), (-5, 5)]
    try:
        result = optimize.minimize(
            neg_log_lik_zipf_alekseev, initial_params, args=(dd_counts, L),
            method='L-BFGS-B', bounds=bounds, options={'maxiter': MAXITER_FIT}
        )
        a_param, b_param = result.x
        converged = result.success
        x_vals = np.arange(1, L + 1)
        log_unnorm = -(a_param + b_param * np.log(x_vals)) * np.log(x_vals)
        unnorm_p = np.exp(log_unnorm)
        expected_p = unnorm_p / np.sum(unnorm_p)
        observed_counts = np.zeros(L)
        for x, count in dd_counts.items():
            if 1 <= x <= L:
                observed_counts[x-1] = count
        observed_cdf = np.cumsum(observed_counts) / total_count
        expected_cdf = np.cumsum(expected_p)
        ks_stat = np.max(np.abs(observed_cdf - expected_cdf))
        ks_p = 1 - stats.kstwo.sf(ks_stat, total_count)
        return {'a_param': a_param, 'b_param': b_param, 'converged': converged,
                'ks_stat': ks_stat, 'ks_p': ks_p, 'L': L, 'n_deps': total_count}
    except Exception as e:
        print(f"ZA fitting failed: {e}")
        return {'a_param': np.nan, 'b_param': np.nan, 'converged': False}

print('ZA distribution functions defined')

In [ ]:
# Exponential distribution fitting (baseline)

def fit_exponential_distribution(dd_counts: Dict[int, int]) -> Dict:
    """Fit exponential distribution as baseline."""
    if not dd_counts or len(dd_counts) < 2:
        return {'lambda_param': np.nan, 'converged': False}
    total_count = sum(dd_counts.values())
    weighted_sum = sum(x * count for x, count in dd_counts.items())
    mean_dd = weighted_sum / total_count
    if mean_dd <= 0:
        return {'lambda_param': np.nan, 'converged': False}
    return {'lambda_param': 1.0 / mean_dd, 'converged': True, 'mean_dd': mean_dd}

print('Exponential distribution function defined')

## Step 5: Spoken vs Written Analysis

Compare dependency distance patterns between spoken and written treebanks.

In [ ]:
# Spoken vs written analysis - from method.py

def analyze_spoken_written(merged_df: pd.DataFrame) -> Dict:
    """Analyze spoken vs written differences in dependency distance."""
    print("Analyzing spoken vs written differences...")
    merged_df['genre_category'] = merged_df['genre'].apply(
        lambda x: 'spoken' if isinstance(x, str) and 'spoken' in x.lower() else 'written'
    )
    treebank_stats = merged_df.groupby(['treebank_name', 'genre_category', 'family']).agg({
        'dd': ['mean', 'count']}).reset_index()
    treebank_stats.columns = ['treebank_name', 'genre_category', 'family', 'mean_dd', 'n_deps']
    spoken_mean_dd = treebank_stats[treebank_stats['genre_category'] == 'spoken']['mean_dd']
    written_mean_dd = treebank_stats[treebank_stats['genre_category'] == 'written']['mean_dd']
    if len(spoken_mean_dd) > 0 and len(written_mean_dd) > 0:
        t_stat, t_p = stats.ttest_ind(spoken_mean_dd, written_mean_dd, equal_var=False)
        cohens_d = (spoken_mean_dd.mean() - written_mean_dd.mean()) / np.sqrt(
            (spoken_mean_dd.var() + written_mean_dd.var()) / 2)
    else:
        t_stat, t_p, cohens_d = np.nan, np.nan, np.nan
    return {
        't_test': {
            'statistic': t_stat, 'p': t_p, 'cohens_d': cohens_d,
            'spoken_mean': spoken_mean_dd.mean() if len(spoken_mean_dd) > 0 else np.nan,
            'written_mean': written_mean_dd.mean() if len(written_mean_dd) > 0 else np.nan
        }
    }

print('analyze_spoken_written function defined')

## Step 6: Mixed-Effects Models

Run mixed-effects models to predict ZA parameters from WALS features, with random intercepts for language family.

In [ ]:
# Mixed-effects model functions - from method.py

def prepare_mixed_effects_data(merged_df: pd.DataFrame, za_results: List[Dict]) -> pd.DataFrame:
    """Prepare data for mixed-effects models."""
    print("Preparing data for mixed-effects models...")
    za_df = pd.DataFrame(za_results)
    treebank_meta = merged_df[['treebank_name', 'wals_1A_value', 'wals_20A_value',
                               'wals_26A_value', 'wals_49A_value', 'wals_51A_value']].drop_duplicates()
    for col in ['wals_1A_value', 'wals_20A_value', 'wals_26A_value', 'wals_49A_value', 'wals_51A_value']:
        treebank_meta[col] = pd.to_numeric(treebank_meta[col], errors='coerce')
    sent_len_df = merged_df.groupby('treebank_name')['sentence_length'].mean().reset_index()
    sent_len_df.columns = ['treebank_name', 'mean_sentence_length']
    model_df = za_df.merge(treebank_meta, on='treebank_name', how='inner')
    model_df = model_df.merge(sent_len_df, on='treebank_name', how='inner')
    model_df = model_df.dropna(subset=['a_param', 'b_param', 'family'])
    print(f"Mixed-effects data shape: {model_df.shape}")
    return model_df

def run_mixed_effects_model(df: pd.DataFrame, outcome: str, predictor: str) -> Dict:
    """Run mixed-effects model for a single predictor."""
    print(f"Running mixed-effects model: {outcome} ~ {predictor}")
    y = df[outcome]
    X = sm.add_constant(df[[predictor, 'mean_sentence_length']])
    groups = df['family']
    try:
        model = MixedLM(y, X, groups=groups)
        result = model.fit()
        params = result.params
        bse = result.bse
        pvalues = result.pvalues
        return {
            'converged': result.converged,
            'predictor_coef': params[predictor],
            'predictor_se': bse[predictor],
            'predictor_p': pvalues[predictor],
            'intercept': params['const'],
            'sentence_length_coef': params['mean_sentence_length'],
            'sentence_length_p': pvalues['mean_sentence_length'],
        }
    except Exception as e:
        print(f"Mixed-effects model failed for {predictor}: {e}")
        return {'converged': False, 'error': str(e)}

print('Mixed-effects functions defined')

## Step 7: Outlier Detection and Robustness Checks

Detect outlier families and perform robustness checks (bootstrap, distribution comparison).

In [ ]:
# Outlier detection and robustness - from method.py

def detect_outliers(model_results: Dict, df: pd.DataFrame) -> List[Dict]:
    """Detect outlier families via random effects."""
    print("Detecting outliers via random effects...")
    outliers = []
    for outcome in ['a_param', 'b_param']:
        family_means = df.groupby('family')[outcome].mean()
        global_mean = df[outcome].mean()
        family_effects = family_means - global_mean
        se = family_effects.std()
        outlier_families = family_effects[abs(family_effects) > 2 * se].index.tolist()
        for family in outlier_families:
            outliers.append({
                'family': family, 'outcome': outcome, 'effect': family_effects[family],
                'se': se, 'n_treebanks': len(df[df['family'] == family])
            })
    return outliers

def robustness_checks(df: pd.DataFrame, za_results: List[Dict]) -> Dict:
    """Perform robustness checks."""
    print("Performing robustness checks...")
    results = {}
    za_df = pd.DataFrame(za_results)
    bootstrap_results = []
    for i in range(N_BOOTSTRAP):
        sampled_treebanks = np.random.choice(
            za_df['treebank_name'].unique(),
            size=int(0.8 * za_df['treebank_name'].nunique()),
            replace=False)
        sampled_df = za_df[za_df['treebank_name'].isin(sampled_treebanks)].copy()
        if len(sampled_df) > 0:
            bootstrap_results.append({
                'a_param_mean': sampled_df['a_param'].mean(),
                'b_param_mean': sampled_df['b_param'].mean()})
    if bootstrap_results:
        bootstrap_df = pd.DataFrame(bootstrap_results)
        results['bootstrap'] = {
            'a_param_mean_ci': [bootstrap_df['a_param_mean'].quantile(0.025),
                                bootstrap_df['a_param_mean'].quantile(0.975)],
            'b_param_mean_ci': [bootstrap_df['b_param_mean'].quantile(0.025),
                                bootstrap_df['b_param_mean'].quantile(0.975)]}
    return results

print('Outlier detection and robustness functions defined')

## Main Execution

Run the full pipeline on the demo data.

In [ ]:
# Main execution - adapted from method.py main()
print("Starting experiment...")

# Extract examples from loaded data
dd_examples = data.get('dd_examples', [])
wals_examples = data.get('wals_examples', [])

# Merge datasets
merged_df = merge_datasets(dd_examples, wals_examples)

# Filter treebanks
filtered_df = filter_treebanks(merged_df, min_deps=MIN_DEPS)
print(f"Filtered DataFrame shape: {filtered_df.shape}")
print(f"Unique treebanks: {filtered_df['treebank_name'].nunique()}")

# STEP 2: ZA distribution fitting
print("\n=== STEP 2: ZA Distribution Fitting ===")
za_results = []
treebanks = filtered_df['treebank_name'].unique()
print(f"Fitting ZA distribution to {len(treebanks)} treebanks...")

for treebank in treebanks:
    treebank_data = filtered_df[filtered_df['treebank_name'] == treebank]
    dd_counts = compute_dd_distribution(treebank_data)
    za_fit = fit_za_distribution(dd_counts)
    za_fit['treebank_name'] = treebank
    za_fit['family'] = treebank_data['family'].iloc[0]
    za_fit['n_deps'] = len(treebank_data)
    za_results.append(za_fit)

print(f"ZA fitting complete. Successful fits: {sum(1 for r in za_results if r.get('converged', False))}")

# STEP 3: Spoken vs written analysis
print("\n=== STEP 3: Spoken vs Written Analysis ===")
spoken_written_results = analyze_spoken_written(merged_df)
if 't_test' in spoken_written_results:
    t_test = spoken_written_results['t_test']
    print(f"Spoken vs written: t={t_test.get('statistic', 'N/A'):.3f}, p={t_test.get('p', 'N/A'):.3f}")

# STEP 4: Mixed-effects models
print("\n=== STEP 4: Mixed-Effects Models ===")
model_df = prepare_mixed_effects_data(filtered_df, za_results)
wals_features = ['wals_1A_value', 'wals_20A_value', 'wals_26A_value', 'wals_49A_value', 'wals_51A_value']
mixed_effects_results = {}
for feature in wals_features:
    model_result = run_mixed_effects_model(model_df, 'a_param', feature)
    mixed_effects_results[feature] = model_result
    if model_result.get('converged'):
        print(f"{feature}: coef={model_result['predictor_coef']:.4f}, p={model_result['predictor_p']:.4f}")

# STEP 5: Outlier detection
print("\n=== STEP 5: Outlier Detection ===")
outliers = detect_outliers(mixed_effects_results, model_df)
print(f"Outliers detected: {len(outliers)}")

# STEP 6: Robustness checks
print("\n=== STEP 6: Robustness Checks ===")
robustness = robustness_checks(filtered_df, za_results)
print("Bootstrap analysis complete")

print("\n=== Experiment Complete ===")

## Results Visualization

Display key results from the experiment.

In [ ]:
# Visualization
print("ZA Distribution Fitting Results:")
print("-" * 50)
for r in za_results:
    if r.get('converged'):
        print(f"Treebank: {r['treebank_name']}, a={r['a_param']:.3f}, b={r['b_param']:.3f}, n_deps={r['n_deps']}")

print("\nMixed-Effects Model Results (a_param ~ WALS features):")
print("-" * 50)
for feature, result in mixed_effects_results.items():
    if result.get('converged'):
        print(f"{feature}: coef={result['predictor_coef']:.4f}, p={result['predictor_p']:.4f}")

# Plot ZA parameters by treebank
if len(za_results) > 0:
    za_df = pd.DataFrame(za_results)
    za_df = za_df[za_df['converged']]
    if len(za_df) > 0:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].bar(range(len(za_df)), za_df['a_param'])
        ax[0].set_xlabel('Treebank')
        ax[0].set_ylabel('a parameter')
        ax[0].set_title('ZA a parameter by treebank')
        ax[0].set_xticks([])
        ax[1].bar(range(len(za_df)), za_df['b_param'])
        ax[1].set_xlabel('Treebank')
        ax[1].set_ylabel('b parameter')
        ax[1].set_title('ZA b parameter by treebank')
        ax[1].set_xticks([])
        plt.tight_layout()
        plt.show()